In [3]:

#Preparación de la estructura (Data Splitting)
import os
import shutil
import random

# montar el entorno de drive - esto es para que me encuentren los ficheros,
from google.colab import drive
drive.mount('/content/drive')

# incorporar el nombre del proyecto
ruta_proyecto = '/content/drive/MyDrive/'
os.chdir(ruta_proyecto)




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Preparación de la estructura (Data Splitting)
def organizar_para_yolo(input_dir, output_dir, split=0.8):
    # Creamos carpetas raíz
    for s in ['train', 'val']:
        os.makedirs(os.path.join(output_dir, 'images', s), exist_ok=True)
        # Nota: YOLO para detección de objetos usa carpetas 'labels',
        # pero como estamos haciendo Clasificación inicial, se usa estructura por carpetas.

    clases = [d for d in os.listdir(input_dir) if os.path.isdir(os.path.join(input_dir, d))]

    for clase in clases:
        ruta_clase = os.path.join(input_dir, clase)
        imagenes = os.listdir(ruta_clase)
        random.shuffle(imagenes)

        limite = int(len(imagenes) * split)
        train_imgs = imagenes[:limite]
        val_imgs = imagenes[limite:]

        # Copiar archivos
        for img in train_imgs:
            os.makedirs(os.path.join(output_dir, 'train', clase), exist_ok=True)
            shutil.copy(os.path.join(ruta_clase, img), os.path.join(output_dir, 'train', clase, img))

        for img in val_imgs:
            os.makedirs(os.path.join(output_dir, 'val', clase), exist_ok=True)
            shutil.copy(os.path.join(ruta_clase, img), os.path.join(output_dir, 'val', clase, img))

    print(f"✅ Dataset organizado en {output_dir}")

# Ejecución
organizar_para_yolo("./fridge_preprocessed", "./yolo_dataset")

In [ ]:
#Entrenamiento del Modelo Local
# Instala la librería necesaria para usar YOLOv8
!pip install ultralytics
from ultralytics import YOLO

# 1. Cargar un modelo base (nano es el más rápido para pruebas locales)
# Usamos 'yolov8n-cls.pt' porque tu dataset está organizado por carpetas (clasificación)
model = YOLO('yolov8n-cls.pt')

# 2. Entrenar el modelo
results = model.train(
    data="./yolo_dataset",
    epochs=25,             # Número de pasadas por el dataset
    imgsz=640,             # Tamaño de imagen (el mismo que tu preprocesado)
    batch=16,              # Ajustar según la memoria de tu GPU/CPU
    name='modelo_frigorifico_v1'
)

print("🚀 Entrenamiento finalizado. Modelo guardado en: runs/classify/modelo_frigorifico_v1/weights/best.pt")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 22.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./yolo_dataset, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None

In [ ]:
#métricas del modelo
# 1. Ejecutar validación sobre el conjunto de test/val
metrics = model.val()

# 2. Extraer parámetros de Precisión
print("===== MÉTRICAS DE PRECISIÓN =====")
print(f"Precisión Top-1 (Exactitud): {metrics.top1 * 100:.2f}%")
print(f"Precisión Top-5: {metrics.top5 * 100:.2f}%")

# 3. Extraer parámetros de Inferencia (Latencia)
# Los tiempos están en milisegundos (ms) por imagen
tiempos = metrics.speed
print("\n===== MÉTRICAS DE LATENCIA (INFERENCIA) =====")
print(f"Pre-procesado: {tiempos['preprocess']:.2f} ms")
print(f"Inferencia (IA): {tiempos['inference']:.2f} ms")
print(f"Post-procesado: {tiempos['postprocess']:.2f} ms")
print(f"TIEMPO TOTAL: {sum(tiempos.values()):.2f} ms por imagen")


Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cpu CPU (Intel Xeon CPU @ 2.20GHz)
YOLOv8n-cls summary (fused): 30 layers, 1,470,748 parameters, 35,868 gradients, 3.3 GFLOPs

WARNING ⚠️ Dataset not found, missing path /content/drive/MyDrive/datasets/imagenet, attempting download...
WARNING ⚠️ Download failure, retrying 1/3 https://ultralytics.com/assets/../datasets/imagenet.zip... HTTP Error 404: Not Found
Dataset download success ✅ (1.6s), saved to /content/drive/MyDrive/datasets/imagenet

WARNING ⚠️ Dataset 'split=train' not found at /content/drive/MyDrive/datasets/imagenet/train
ERROR ❌ No images found in /content/drive/MyDrive/datasets/imagenet or its subdirectories.
WARNING ⚠️ Dataset 'split=val' not found, using 'split=test' instead.


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/datasets/imagenet/train'